In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

In [2]:
icu_file_path = '/home/lu.kuang/thesis/Prospective_DataCleaning/DATA_v2.xlsx'

In [3]:
# Creating PRESCRIPTIONS table
PRESCRIPTIONS = pd.DataFrame(columns = ['ROW_ID',
                                        'PRESCRIPTIONS_ID',
                                        'ADMISSION_ID',
                                        'PATIENT_ID',
                                        'ITEM_ID',
                                        'START_DATETIME',
                                        'END_DATETIME',
                                        'DELTA_TIME',
                                        'DELTA_UOM',
                                        'VALUE_NUM',
                                        'VALUE_UOM',
                                        'RATE_NUM',
                                        'RATE_UOM'])

In [4]:
# Creating ITEMS table
ITEMS = pd.DataFrame(columns = ['ROW_ID',
                                'ITEM_ID',
                                'LABEL',
                                'ABBREVIATION',
                                'CATEGORY',
                                'PARAM_TYPE',
                                'UOM'])

# getting reference count from measurements
count_ref_item = int(pd.read_pickle('ITEMS_INTERVENTIONS.zip')['ITEM_ID'].max()[1:]) + 1
count_ref_item 

300

In [5]:
# Opening ADMISSIONS for ADMISSION_ID, PATIENT_ID, RECORD_ID reference
ADMISSIONS = pd.read_csv('ADMISSIONS.csv',  delimiter = ";", encoding = 'Latin-1')
ADMISSIONS = ADMISSIONS[['ADMISSION_ID', 'PATIENT_ID', 'RECORD_ID']]
ADMISSIONS

,ADMISSION_ID,PATIENT_ID,RECORD_ID
0,A00000,P00033,189844
1,A00001,P00033,190055
2,A00002,P00005,189946
3,A00003,P00005,209493
4,A00004,P00169,190072
...,...,...,...
296,A00296,P00268,210081
297,A00297,P00182,210086
298,A00298,P00056,210095
299,A00299,P00253,210098


In [6]:
###############################################################################################################################
###############################################################################################################################

In [7]:
# preparing Meds for inclusion
data = pd.read_excel(io = icu_file_path, sheet_name = 'MEDS')
data

,RECORD_ID,SITE,MRN,ICU_ADMIT,ParamABB,chartdate,enddate,TimeDuration_MI,Dose,Dose_Unit,Rate,Rate_Unit,Weight_KG
0,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 02:16:00,2021-08-01 02:30:00,14,186.67,mg,80.71,mcg/kg/min,165.2
1,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 02:30:00,2021-08-01 03:10:00,40,331.97,mg,50.24,mcg/kg/min,165.2
2,189844,FMC ICU,1005018831,2021-08-01 02:14:00,FentaNYL,2021-08-01 02:41:00,2021-08-01 04:40:00,119,99.17,mcg,50.00,mcg/hour,165.2
3,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 03:10:00,2021-08-01 04:10:00,60,401.13,mg,40.47,mcg/kg/min,165.2
4,189844,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 04:10:00,2021-08-01 04:26:00,16,80.23,mg,30.35,mcg/kg/min,165.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
20014,210099,FMC ICU,1005499197,2022-07-27 20:56:00,Propofol,2022-07-31 08:25:00,2022-07-31 09:01:00,36,193.55,mg,50.01,mcg/kg/min,107.5
20015,210099,FMC ICU,1005499197,2022-07-27 20:56:00,Propofol,2022-07-31 09:01:00,2022-07-31 10:39:00,98,316.56,mg,30.05,mcg/kg/min,107.5
20016,210099,FMC ICU,1005499197,2022-07-27 20:56:00,Ketamine,2022-07-31 14:40:00,2022-08-01 06:10:00,930,500.00,mg,0.30,mg/kg/hour,107.5
20017,210099,FMC ICU,1005499197,2022-07-27 20:56:00,Ketamine,2022-08-01 06:10:00,2022-08-01 21:40:00,930,500.00,mg,0.30,mg/kg/hour,107.5


In [8]:
# Deleting Weight_KG, as this information is available on table ADMISSIONS
data.drop(['Weight_KG'], axis = 1, inplace = True)

In [9]:
# merging data and ADMISSIONS
data = data.merge(right = ADMISSIONS, on = ['RECORD_ID'])
data.drop(['RECORD_ID'], axis = 1, inplace = True)

In [10]:
data.rename(columns={'ParamABB' : 'ITEM_ID',
                     'chartdate' : 'START_DATETIME',
                     'enddate' : 'END_DATETIME',
                     'TimeDuration_MI' : 'DELTA_TIME',
                     'Dose' : 'VALUE_NUM',
                     'Dose_Unit' : 'VALUE_UOM',
                     'Rate' : 'RATE_NUM',
                     'Rate_Unit' : 'RATE_UOM'}, 
              inplace = True)
data['DELTA_UOM'] = 'min'
data['PRESCRIPTIONS_ID'] = range(len(data))
data['PRESCRIPTIONS_ID'] = data['PRESCRIPTIONS_ID'].apply(lambda x: 'PR'+str(x).rjust(8, '0'))
data.head()

,SITE,MRN,ICU_ADMIT,ITEM_ID,START_DATETIME,END_DATETIME,DELTA_TIME,VALUE_NUM,VALUE_UOM,RATE_NUM,RATE_UOM,ADMISSION_ID,PATIENT_ID,DELTA_UOM,PRESCRIPTIONS_ID
0,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 02:16:00,2021-08-01 02:30:00,14,186.67,mg,80.71,mcg/kg/min,A00000,P00033,min,PR00000000
1,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 02:30:00,2021-08-01 03:10:00,40,331.97,mg,50.24,mcg/kg/min,A00000,P00033,min,PR00000001
2,FMC ICU,1005018831,2021-08-01 02:14:00,FentaNYL,2021-08-01 02:41:00,2021-08-01 04:40:00,119,99.17,mcg,50.00,mcg/hour,A00000,P00033,min,PR00000002
3,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 03:10:00,2021-08-01 04:10:00,60,401.13,mg,40.47,mcg/kg/min,A00000,P00033,min,PR00000003
4,FMC ICU,1005018831,2021-08-01 02:14:00,Propofol,2021-08-01 04:10:00,2021-08-01 04:26:00,16,80.23,mg,30.35,mcg/kg/min,A00000,P00033,min,PR00000004


In [11]:
# appending to INTERVENTIONS_data
PRESCRIPTIONS_data = pd.concat([PRESCRIPTIONS, data], axis = 0, ignore_index = True)
# Identifying the max value of PRESCRIPTIONS_ID
count_ref_prescription = int(PRESCRIPTIONS_data['PRESCRIPTIONS_ID'].max()[2:]) + 1

In [12]:
# preparing ITEM_ID info
temp = pd.DataFrame(PRESCRIPTIONS_data['ITEM_ID'].unique(), columns = ['LABEL'])
temp['ROW_ID'] = range(count_ref_item, count_ref_item + len(temp))
temp['ITEM_ID'] = temp['ROW_ID'].apply(lambda x: 'I'+str(x).rjust(3, '0'))

# Creating dict using original param labels
item_dict = temp[['LABEL', 'ITEM_ID']].set_index(['LABEL']).to_dict()

# continuing with ITEM_ID info preparation
temp['ABBREVIATION'] = temp['LABEL']
temp['CATEGORY'] = 'Prescriptions'
temp['UOM'] = temp.apply(lambda x: PRESCRIPTIONS_data[PRESCRIPTIONS_data['ITEM_ID'] == x['LABEL']]['VALUE_UOM'].values[0], axis = 1)

temp['PARAM_TYPE'] = 'NUM'

In [13]:
# Adjusting data using item_dict
PRESCRIPTIONS_data['LABEL'] = PRESCRIPTIONS_data['ITEM_ID']
PRESCRIPTIONS_data['ITEM_ID'].replace(item_dict['ITEM_ID'], inplace = True)
PRESCRIPTIONS_data

,ROW_ID,PRESCRIPTIONS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,START_DATETIME,END_DATETIME,DELTA_TIME,DELTA_UOM,VALUE_NUM,VALUE_UOM,RATE_NUM,RATE_UOM,SITE,MRN,ICU_ADMIT,LABEL
0,NaN,PR00000000,A00000,P00033,I300,2021-08-01 02:16:00,2021-08-01 02:30:00,14,min,186.67,mg,80.71,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
1,NaN,PR00000001,A00000,P00033,I300,2021-08-01 02:30:00,2021-08-01 03:10:00,40,min,331.97,mg,50.24,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
2,NaN,PR00000002,A00000,P00033,I301,2021-08-01 02:41:00,2021-08-01 04:40:00,119,min,99.17,mcg,50.00,mcg/hour,FMC ICU,1.005019e+09,2021-08-01 02:14:00,FentaNYL
3,NaN,PR00000003,A00000,P00033,I300,2021-08-01 03:10:00,2021-08-01 04:10:00,60,min,401.13,mg,40.47,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
4,NaN,PR00000004,A00000,P00033,I300,2021-08-01 04:10:00,2021-08-01 04:26:00,16,min,80.23,mg,30.35,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20086,NaN,PR00020086,A00300,P00036,I300,2022-07-31 08:25:00,2022-07-31 09:01:00,36,min,193.55,mg,50.01,mcg/kg/min,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Propofol
20087,NaN,PR00020087,A00300,P00036,I300,2022-07-31 09:01:00,2022-07-31 10:39:00,98,min,316.56,mg,30.05,mcg/kg/min,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Propofol
20088,NaN,PR00020088,A00300,P00036,I308,2022-07-31 14:40:00,2022-08-01 06:10:00,930,min,500.00,mg,0.30,mg/kg/hour,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Ketamine
20089,NaN,PR00020089,A00300,P00036,I308,2022-08-01 06:10:00,2022-08-01 21:40:00,930,min,500.00,mg,0.30,mg/kg/hour,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Ketamine


In [14]:
# appending ITEMS
ITEMS_data = pd.concat([ITEMS, temp], axis = 0, ignore_index = True)
# Identifying the max value of ITEM_ID
count_ref_item = int(ITEMS_data['ITEM_ID'].max()[1:]) + 1

In [15]:
# saving table to pickles
PRESCRIPTIONS_data.to_pickle('PRESCRIPTIONS_01.pickle')
ITEMS_data.to_pickle('PRESCRIPTIONS_ITEMS_01.pickle')

In [16]:
PRESCRIPTIONS_data

,ROW_ID,PRESCRIPTIONS_ID,ADMISSION_ID,PATIENT_ID,ITEM_ID,START_DATETIME,END_DATETIME,DELTA_TIME,DELTA_UOM,VALUE_NUM,VALUE_UOM,RATE_NUM,RATE_UOM,SITE,MRN,ICU_ADMIT,LABEL
0,NaN,PR00000000,A00000,P00033,I300,2021-08-01 02:16:00,2021-08-01 02:30:00,14,min,186.67,mg,80.71,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
1,NaN,PR00000001,A00000,P00033,I300,2021-08-01 02:30:00,2021-08-01 03:10:00,40,min,331.97,mg,50.24,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
2,NaN,PR00000002,A00000,P00033,I301,2021-08-01 02:41:00,2021-08-01 04:40:00,119,min,99.17,mcg,50.00,mcg/hour,FMC ICU,1.005019e+09,2021-08-01 02:14:00,FentaNYL
3,NaN,PR00000003,A00000,P00033,I300,2021-08-01 03:10:00,2021-08-01 04:10:00,60,min,401.13,mg,40.47,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
4,NaN,PR00000004,A00000,P00033,I300,2021-08-01 04:10:00,2021-08-01 04:26:00,16,min,80.23,mg,30.35,mcg/kg/min,FMC ICU,1.005019e+09,2021-08-01 02:14:00,Propofol
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20086,NaN,PR00020086,A00300,P00036,I300,2022-07-31 08:25:00,2022-07-31 09:01:00,36,min,193.55,mg,50.01,mcg/kg/min,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Propofol
20087,NaN,PR00020087,A00300,P00036,I300,2022-07-31 09:01:00,2022-07-31 10:39:00,98,min,316.56,mg,30.05,mcg/kg/min,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Propofol
20088,NaN,PR00020088,A00300,P00036,I308,2022-07-31 14:40:00,2022-08-01 06:10:00,930,min,500.00,mg,0.30,mg/kg/hour,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Ketamine
20089,NaN,PR00020089,A00300,P00036,I308,2022-08-01 06:10:00,2022-08-01 21:40:00,930,min,500.00,mg,0.30,mg/kg/hour,FMC ICU,1.005499e+09,2022-07-27 20:56:00,Ketamine


In [17]:
PRESCRIPTIONS_data.to_csv('PRESCRIPTIONS.csv', sep = ";", encoding = 'Latin-1')

In [18]:
################################################################################################################################

In [19]:
ITEMS = pd.read_pickle('ITEMS_INTERVENTIONS.zip').reset_index()
PRESCRIPTIONS_ITEMS_01 = pd.read_pickle('PRESCRIPTIONS_ITEMS_01.pickle')
PRESCRIPTIONS_ITEMS = pd.concat([ITEMS,
                                 PRESCRIPTIONS_ITEMS_01,
                                ], axis = 0, ignore_index = True)

del ITEMS
del PRESCRIPTIONS_ITEMS_01

PRESCRIPTIONS_ITEMS.set_index(['ROW_ID'], inplace = True)

PRESCRIPTIONS_ITEMS.to_pickle('ITEMS_PRESCRIPTIONS.pickle')